
# HTML, CSS et inspection d'une page web

**Site étudié :** [Books to Scrape](https://books.toscrape.com/)  
**Catégorie sélectionnée :** Nonfiction

## Étape 1 — Périmètre de collecte

| Indicateur | Valeur |
|---|---:|
| Catégorie | Nonfiction |
| Nombre de livres annoncé | 110 |
| Nombre de pages | 6 |
| Nombre maximal de livres par page | 20 |

La catégorie contient 110 livres répartis sur 6 pages.

## Étape 2 — Identification des fiches produits

Chaque livre est représenté par un élément HTML possédant la classe `product_pod`.

**Sélecteur CSS :** `.product_pod`

**Commande de vérification :**

```javascript
document.querySelectorAll('.product_pod').length
```

**Résultats :**

| Page | Nombre de fiches |
|---|---:|
| 1 | 20 |
| 2 | 20 |
| 3 | 20 |
| 4 | 20 |
| 5 | 20 |
| 6 | 10 |
| **Total** | **110** |

**Contrôle de cohérence :**

20 + 20 + 20 + 20 + 20 + 10 = **110 livres**

Le nombre de fiches identifiées correspond au nombre de livres annoncé par le site.

## Étape 3 — Identification des données à extraire

| Donnée | Sélecteur CSS | Source de la valeur |
|---|---|---|
| Titre | `a[title]` | Attribut `title` |
| Prix | `p.price_color` | Contenu texte |
| Disponibilité | `.in_stock` | Contenu texte |
| Note | `.star-rating` | Attribut `class` |
| Lien vers la fiche | `a[href]` | Attribut `href` |

**Remarque :** les sélecteurs permettent d'identifier les éléments HTML. L'extraction de leurs valeurs nécessite ensuite de lire leur contenu texte ou l'attribut correspondant.

## Étape 4 — Vérification des sélecteurs

Les sélecteurs suivants ont été testés dans la console du navigateur, sur la première page de la catégorie.

**Nombre de fiches produits :**

```javascript
document.querySelectorAll('.product_pod').length
// Résultat : 20
```

**Nombre de prix :**

```javascript
document.querySelectorAll('article.product_pod p.price_color').length
// Résultat : 20
```

**Nombre de notes :**

```javascript
document.querySelectorAll('article.product_pod .star-rating').length
// Résultat : 20
```

### Bilan des contrôles

| Élément | Nombre obtenu | Résultat |
|---|---:|---|
| Fiches produits | 20 | Conforme |
| Prix | 20 | Conforme |
| Notes | 20 | Conforme |

Les trois sélecteurs retournent le même nombre d'éléments.

Les contrôles de cardinalité sont cohérents avec les 20 livres affichés sur cette page.

## Étape 5 — Pagination

**Sélecteur du lien vers la page suivante :** `li.next a`

**Valeur de l'attribut `href` sur la première page :**

```text
page-2.html
```

Le lien vers la page suivante est relatif à l'URL de la page courante.

**Condition d'arrêt :**

Lorsque le lien vers la page suivante est absent, le script doit arrêter la pagination : il a atteint la dernière page.

L'absence de ce lien constitue une condition normale de fin de collecte.

## Étape 6 — Vérification du code source

Le prix d'un livre est présent dans le code source HTML de la page.

**Conséquence :**

La donnée peut être extraite directement à partir du HTML téléchargé, sans nécessiter l'exécution de JavaScript.

## Étape 7 — Informations supplémentaires

La fiche détaillée d'un livre contient des informations absentes de la liste des produits.

Deux informations ont été identifiées :

| Information | Emplacement |
|---|---|
| Taxe (`Tax`) | Tableau des informations produit |
| Quantité en stock (`Availability`) | Tableau des informations produit |

La liste des produits indique uniquement la disponibilité du livre, tandis que la fiche détaillée permet de connaître la quantité en stock.

In [ ]:
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup


In [ ]:
# Constantes de configuration du scraper
_URL = "https://books.toscrape.com/catalogue/category/books/nonfiction_13/page-1.html"
_ENTETES = {"User-Agent": "bootcamp-data/1.0 (Wild Code School)"}
_ENCODING = "utf-8"
_NOTES = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

In [ ]:
class ErreurHttp(Exception):
    def __init__(self, code_erreur):
        super().__init__(f"Server returned status {code_erreur}")
        self.code_erreur = code_erreur

In [ ]:
def lire_une_fiche(fiche):
    dict_fiche = {}
    lien_titre = fiche.select_one("h3 a")
    rating = fiche.select_one("p.star-rating")["class"]

    dict_fiche['prix'] = fiche.select_one("p.price_color").get_text(strip=True)
    dict_fiche['titre'] = lien_titre["title"]
    dict_fiche['url'] = lien_titre["href"]
    dict_fiche['note'] = _NOTES[rating[1]]

    return dict_fiche


def recuperer_une_page(soup):
    # r = requests.get(url, headers=_ENTETES, timeout=10)
    # r.encoding = encoding

    # if r.status_code != 200:
    #    raise ErreurHttp(r.status_code)
    # soup = BeautifulSoup(r.text, "html.parser")
    fiches_livres = soup.select("article.product_pod")
    
    liste_fiche =[]
    for fiche in fiches_livres:
        liste_fiche.append(lire_une_fiche(fiche))

    df = pd.DataFrame(liste_fiche)

    # suivant = soup.select_one("li.next a")
    # url_suivant = urljoin(url, suivant["href"]) if suivant else None

    return df# , url_suivant

In [ ]:
def recuperer_soup(url):
    request = requests.get(url, headers=_ENTETES, timeout=10)
    request.encoding = _ENCODING

    if request.status_code != 200:
        raise ErreurHttp(request.status_code)
    soup = BeautifulSoup(request.text, "html.parser")

    return soup


def collecter_toutes_pages(url_depart, pages_max=10):
    url = url_depart
    df_livres = None
    page = 1

    while url and page <= pages_max:
        soup = recuperer_soup(url)

        df_recuperé = recuperer_une_page(soup)
        df_recuperé['url'] = df_recuperé['url'].apply(lambda x: urljoin(url, x))
        df_livres = pd.concat([df_livres, df_recuperé], ignore_index=True)
        print(f"Page {page} : {df_recuperé.shape[0]} livres (total {df_livres.shape[0]})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        page += 1
        time.sleep(1)
    
    return df_livres
        
        
df_livres = collecter_toutes_pages(_URL, 12)
# Test en dur
print(f"{df_livres.shape[0]} (attendu 110) : {df_livres.shape[0] == 110}")

In [ ]:
def recuperer_stock(availability):

    if not availability:
        return None

    trouve = re.search(r"\((\d+) available\)", availability)
    return int(trouve.group(1)) if trouve else 0


def fiche_detaillee(soup):   

    informations_produit = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        valeur = ligne.select_one("td")
        if cle and valeur:
            informations_produit[cle.get_text(strip=True)] = valeur.get_text(strip=True)

    fil_ariane = soup.select("ul.breadcrumb li a")
    categorie = fil_ariane[-1].get_text(strip=True) if fil_ariane else None

    details_livre = {}
    details_livre['upc'] = informations_produit.get("UPC")
    details_livre['stock'] = recuperer_stock(informations_produit.get("Availability"))
    details_livre['categorie'] = categorie

    return details_livre

def recuperer_details(df):
    details = []
    for row in df.itertuples():
        soup = recuperer_soup(row.url)
        details_livre = fiche_detaillee(soup)
        details_livre['url'] = row.url
        details.append(details_livre)
        print(f"Fiche détaillée de {row.titre} traitée")
        time.sleep(1)

    return details


liste_details_livres = recuperer_details(df_livres)

In [ ]:
df_details_livres = pd.DataFrame(liste_details_livres)
df_total = pd.merge(df_livres, df_details_livres, how="inner", left_on='url', right_on='url')

df_total['prix'] = df_total['prix'].str.replace('£', '')
df_total['prix'] = df_total['prix'].astype(float)
df_total = df_total.rename(columns={'prix': 'prix_gbp'})

df_total = df_total[[
    'titre',
    'categorie',
    'prix_gbp',
    'note',
    'stock',
    'upc',
    'url'
    ]]

In [ ]:
# Vérification de la jointure des données

url_livres = set(df_livres['url'])
url_details = set(x['url'] for x in liste_details_livres)

print("1. Cardinalité des collectes :",
      len(liste_details_livres) == df_livres.shape[0])

print("2. Correspondance des URL :",
      url_livres == url_details)

print("3. Unicité des URL des livres :",
      len(url_livres) == df_livres.shape[0])

print("4. Unicité des URL des fiches détaillées :",
      len(url_details) == len(liste_details_livres))

print("5. Cardinalité après jointure :",
      df_total.shape[0] == df_livres.shape[0])

print("6. Absence de valeurs manquantes :",
      df_total.isna().sum().sum() == 0)

In [ ]:
nombre_livre = df_total.shape[0]
stock_total = df_total['stock'].sum()
prix_moyen = df_total['prix_gbp'].mean()
categorie_distinct = df_total['categorie'].nunique()
print("========== ANALYSE DES LIVRES ==========\n")

print(f"Nombre de livres : {nombre_livre}")
print(f"Stock total (exemplaires) : {stock_total}")
print(f"Prix moyen : {prix_moyen:.2f} £")
print(f"Nombre de catégories distinctes : {categorie_distinct}")
print("-"*40)
print("--- Livre le plus cher ---")
print(df_total.nlargest(1, "prix_gbp").iloc[0])

print("-"*40)
print("--- Répartition des notes (%) ---")
print(round(df_total.groupby('note').size() / nombre_livre * 100.0, 2))

print("\n========== FIN DE L'ANALYSE ==========")

Il n'y a qu'une seule catégorie différente dans le dataframe, la collecte est donc cohérente avec le périmètre prévu

In [ ]:
nom_fichier = df_total.iloc[0]['categorie'] + ".csv"
dossier_data = Path.cwd().parent / "data"
dossier_data.mkdir(parents=True, exist_ok=True)
chemin_csv = dossier_data / nom_fichier
df_total.to_csv(chemin_csv, index=False)